# Extract Record IDs from NCBI accessions
Extract record IDs from NCBI accessions using `assembly_data_report.csv`.

For each accession in the CSV, it looks for the genomic FASTA file in:
`viral_data_all/ncbi_dataset/data_subset/<accession>/` or `viral_data_all/ncbi_dataset/data/<accession>/`
and extracts the record IDs from the FASTA headers.

Output: CSV with columns `accession`, `record_id`
NAME: accession_record_table.csv


In [1]:
import sys
import pandas as pd
from pathlib import Path

def extract_record_ids_from_fasta(fasta_path: Path) -> list[str]:
    """Extract sequence record IDs from a FASTA file's headers."""
    records = []
    with open(fasta_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith(">"):
                # Usually: ">MG982782.1 Influenza A virus..."
                # The record ID is everything after '>' up to the first space
                record_id = line.strip()[1:].split(maxsplit=1)[0]
                records.append(record_id)
    return records

In [3]:
# Parameters
input_file = "../tables/assembly_data_report.csv"
output_file = "../tables/accession_record_table.csv"
dataset_dirs = [
    "../viral_data_all/ncbi_dataset/data_subset",
    "../viral_data_all/ncbi_dataset/data"
]
skip_empty = False

input_path = Path(input_file)
if not input_path.exists():
    raise FileNotFoundError(f"[ERR] Input file not found: {input_path}")

In [4]:
# Read accessions using pandas
df = pd.read_csv(input_path)
if "accession" not in df.columns:
    raise ValueError("[ERR] Input CSV must have an 'accession' column.")

accessions = df["accession"].dropna().astype(str).unique().tolist()
dataset_dirs_paths = [Path(d) for d in dataset_dirs]

records_data = []

for accession in accessions:
    acc_dir = None
    for base_dir in dataset_dirs_paths:
        candidate = base_dir / accession
        if candidate.exists() and candidate.is_dir():
            acc_dir = candidate
            break

    if not acc_dir:
        # Silently skip missing accessions to avoid console spam
        if not skip_empty:
            records_data.append({"accession": accession, "record_id": pd.NA})
        continue

    # Look for fna files
    fna_files = list(acc_dir.glob("*.fna"))
    
    # Strictly exclude derived files to ensure we ONLY use the full sequence file
    genomic_files = [
        f for f in fna_files 
        if not f.name.startswith("cds_") 
        and not f.name.startswith("rna_") 
        and not f.name.startswith("protein_")
    ]

    records = []
    for fna_path in genomic_files:
        records.extend(extract_record_ids_from_fasta(fna_path))

    # Deduplicate while preserving order
    seen = set()
    unique = []
    for r in records:
        if r not in seen:
            seen.add(r)
            unique.append(r)
    
    if unique:
        for rec_id in unique:
            records_data.append({"accession": accession, "record_id": rec_id})
    else:
        if not skip_empty:
            records_data.append({"accession": accession, "record_id": pd.NA})

# Output using pandas
out_df = pd.DataFrame(records_data)
out_df.to_csv(output_file, index=False)
print(f"Extraction complete. Results saved to {output_file}")

Extraction complete. Results saved to ../tables/accession_record_table.csv
